<a href="https://colab.research.google.com/github/vrinda07-ui/gridlock/blob/main/gridboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install CatBoost
!pip install catboost -q

# Imports
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load datasets
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# Separate features and target
X = train.drop(columns=['demand'])
y = train['demand']

# Categorical columns
cat_features = [
    'geohash',
    'timestamp',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

# Convert categorical columns to string
for col in cat_features:
    X[col] = X[col].astype(str)
    test[col] = test[col].astype(str)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Create model
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',
    eval_metric='R2',
    verbose=100
)

# Train
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

# Validation score
val_pred = model.predict(X_val)
r2 = r2_score(y_val, val_pred)

print("\nValidation R²:", r2)
print("Competition Score Approx:", r2 * 100)

# Train on full data
model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=100
)

# Predict test set
test_pred = model.predict(test)

# Create submission
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_pred
})

submission.to_csv('submission.csv', index=False)

print("submission.csv created successfully!")
submission.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00
Train Shape: (77299, 11)
Test Shape: (41778, 10)
0:	learn: 0.0440785	test: 0.0448155	best: 0.0448155 (0)	total: 114ms	remaining: 3m 47s
100:	learn: 0.9016594	test: 0.9052364	best: 0.9052364 (100)	total: 10.2s	remaining: 3m 11s
200:	learn: 0.9230646	test: 0.9220249	best: 0.9220249 (200)	total: 22s	remaining: 3m 17s
300:	learn: 0.9306608	test: 0.9273598	best: 0.9273598 (300)	total: 33.5s	remaining: 3m 9s
400:	learn: 0.9364529	test: 0.9313400	best: 0.9313400 (400)	total: 45.2s	remaining: 3m
500:	learn: 0.9407529	test: 0.9341009	best: 0.9341009 (500)	total: 56.8s	remaining: 2m 50s
600:	learn: 0.9448647	test: 0.9366380	best: 0.9366435 (599)	total: 1m 8s	remaining: 2m 39s
700:	learn: 0.9476470	test: 0.9379685	best: 0.9379685 (700)	total: 1m 20s	remaining: 2m 28s
800:	learn: 0.9503853	test: 0.9393802	best: 0.9393802 (800)	total: 1m 32s	remaining: 2m 17s
900:	learn: 0.9525813	test: 0.9405981	best: 0.9405981 (900)	tot

,Index,demand
0,0,0.045256
1,1,0.019677
2,2,0.005475
3,3,0.027622
4,4,0.039519


In [ ]:
# ... [Keep your first train and validation code exactly the same] ...

# 1. Get the exact best iteration from the validation run
best_iter = model.get_best_iteration()
print(f"\nRetraining final model using optimal iterations: {best_iter}")

# 2. Instantiate a FRESH model using that exact iteration count
final_model = CatBoostRegressor(
    iterations=best_iter,        # Only trains up to the exact sweet spot
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',
    verbose=100
)

# 3. Train on the entire dataset (X, y)
final_model.fit(X, y, cat_features=cat_features)

# 4. Predict test set using the optimized final model
test_pred = final_model.predict(test)

# Create submission
submission = pd.DataFrame({
    'Index': test['Index'], # Ensure 'Index' is exactly what the sample submission expects!
    'demand': test_pred
})

submission.to_csv('submission.csv', index=False)
print("Optimized submission.csv created successfully!")


Retraining final model using optimal iterations: None
0:	learn: 0.1390149	total: 475ms	remaining: 7m 54s
100:	learn: 0.0430256	total: 10.3s	remaining: 1m 31s
200:	learn: 0.0384888	total: 22.9s	remaining: 1m 31s
300:	learn: 0.0370485	total: 34s	remaining: 1m 19s
400:	learn: 0.0357502	total: 45.7s	remaining: 1m 8s
500:	learn: 0.0343312	total: 59.2s	remaining: 59s
600:	learn: 0.0332521	total: 1m 11s	remaining: 47.8s
700:	learn: 0.0323664	total: 1m 25s	remaining: 36.3s
800:	learn: 0.0315730	total: 1m 38s	remaining: 24.5s
900:	learn: 0.0308817	total: 1m 51s	remaining: 12.3s
999:	learn: 0.0303659	total: 2m 4s	remaining: 0us
Optimized submission.csv created successfully!


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Calculate Standard Regression Metrics
val_r2 = r2_score(y_val, val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_mae = mean_absolute_error(y_val, val_pred)

print("--- Validation Performance Metrics ---")
print(f"R² Score (Coefficient of Determination): {val_r2:.5f}")
print(f"RMSE (Root Mean Squared Error):        {val_rmse:.5f}")
print(f"MAE (Mean Absolute Error):             {val_mae:.5f}")

# 2. Check for Systematic Prediction Bias
# (Tells you if your model is systematically over-guessing or under-guessing)
mean_error = np.mean(val_pred - y_val)
print(f"Mean Prediction Bias:                  {mean_error:.5f} (Ideal is 0.0)")

--- Validation Performance Metrics ---
R² Score (Coefficient of Determination): 0.94656
RMSE (Root Mean Squared Error):        0.03288
MAE (Mean Absolute Error):             0.02138
Mean Prediction Bias:                  -0.00018 (Ideal is 0.0)


In [ ]:
# Print summary statistics of your final test predictions
submission_df = pd.DataFrame({'demand': test_pred})
print("--- Test Prediction Statistics ---")
print(submission_df.describe())

print("\n--- Original Train Target Statistics ---")
print(y.describe())

--- Test Prediction Statistics ---
             demand
count  41778.000000
mean       0.115985
std        0.154213
min        0.002320
25%        0.025733
50%        0.056701
75%        0.118502
max        1.011966

--- Original Train Target Statistics ---
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [ ]:
print(train.nunique())

Index            77299
geohash           1249
day                  2
timestamp           96
demand           76715
RoadType             3
NumberofLanes        5
LargeVehicles        2
Landmarks            2
Temperature      74804
Weather              4
dtype: int64


In [ ]:
train[['day']].value_counts()
print(sorted(train['day'].unique()))

[np.int64(48), np.int64(49)]


In [ ]:
temp = train.groupby(
    ['geohash','day','timestamp']
)['demand'].agg(['count','mean','std'])

print(temp.head())
print("Rows:", len(temp))
print("Train rows:", len(train))

                       count      mean  std
geohash day timestamp                      
qp02yc  48  10:30          1  0.046790  NaN
            10:45          1  0.021158  NaN
            1:0            1  0.005397  NaN
            2:30           1  0.012944  NaN
            2:45           1  0.025961  NaN
Rows: 77299
Train rows: 77299


In [ ]:
dup = train.groupby(
    ['geohash','day','timestamp']
)['demand'].nunique()

print("Max unique demand:", dup.max())
print("Groups with >1 demand:",
      (dup > 1).sum())

Max unique demand: 1
Groups with >1 demand: 0


In [ ]:
print(sorted(test['day'].unique()))
print(test['day'].value_counts())

[np.int64(49)]
day
49    41778
Name: count, dtype: int64


In [ ]:
train_keys = set(
    zip(train['geohash'],
        train['day'],
        train['timestamp'])
)

test_keys = set(
    zip(test['geohash'],
        test['day'],
        test['timestamp'])
)

print("Overlap:",
      len(train_keys & test_keys))

print("Test rows:",
      len(test_keys))

Overlap: 0
Test rows: 41778


In [ ]:
print(train['timestamp'].nunique())
print(train['timestamp'].unique()[:10])

96
['0:0' '0:15' '0:30' '0:45' '1:0' '1:15' '1:30' '1:45' '2:0' '2:15']


In [ ]:
importance = model.get_feature_importance()

for feat, imp in zip(X.columns, importance):
    print(feat, imp)

Index 8.110142094274176
geohash 16.921394905243012
day 0.029247117399668624
timestamp 4.988691816051937
RoadType 53.14694968632057
NumberofLanes 5.686777465992087
LargeVehicles 8.948693092142344
Landmarks 0.25050176536289304
Temperature 0.7876946072121406
Weather 1.1299074500011708


In [ ]:
!pip install lightgbm
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=10,
    random_state=42
)



In [ ]:
X = train.drop(columns=['demand', 'Index'])
test_X = test.drop(columns=['Index'])

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = train.drop(columns=['demand', 'Index'])
y = train['demand']

# Encode categorical columns
cat_cols = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'timestamp'
]

for col in cat_cols:
    X[col] = X[col].astype('category')

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("R2:", r2_score(y_val, pred))

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003408 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1441
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 9
[LightGBM] [Info] Start training from score 0.093784
R2: 0.9492846669723884


In [ ]:
print(train.groupby('RoadType')['demand'].describe())
print(train.groupby('Weather')['demand'].describe())
print(train.groupby('day')['demand'].describe())

               count      mean       std           min       25%       50%  \
RoadType                                                                     
Highway       3560.0  0.610756  0.229419  3.500089e-01  0.413629  0.526432   
Residential  69230.0  0.057209  0.052057  6.245650e-07  0.016203  0.040517   
Street        3909.0  0.273164  0.036693  2.200159e-01  0.240702  0.268124   

                  75%       max  
RoadType                         
Highway      0.790095  1.000000  
Residential  0.084468  0.219997  
Street       0.301890  0.349908  
           count      mean       std           min       25%       50%  \
Weather                                                                  
Foggy    20243.0  0.093372  0.141110  6.138162e-06  0.018215  0.047571   
Rainy    20824.0  0.094471  0.141608  7.981559e-07  0.018083  0.048467   
Snowy     7718.0  0.092581  0.138300  3.994529e-06  0.018214  0.047775   
Sunny    27717.0  0.094247  0.144367  6.245650e-07  0.018258  0.04723

In [ ]:
import pandas as pd

train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

In [ ]:
# Geohash mean demand
geo_mean = train.groupby('geohash')['demand'].mean()

train['geo_mean'] = train['geohash'].map(geo_mean)
test['geo_mean'] = test['geohash'].map(geo_mean)

# Timestamp mean demand
time_mean = train.groupby('timestamp')['demand'].mean()

train['time_mean'] = train['timestamp'].map(time_mean)
test['time_mean'] = test['timestamp'].map(time_mean)

# RoadType mean demand
road_mean = train.groupby('RoadType')['demand'].mean()

train['road_mean'] = train['RoadType'].map(road_mean)
test['road_mean'] = test['RoadType'].map(road_mean)

# Geohash + Timestamp mean demand
geo_time_mean = train.groupby(
    ['geohash', 'timestamp']
)['demand'].mean()

train['geo_time_mean'] = train.set_index(
    ['geohash', 'timestamp']
).index.map(geo_time_mean)

test['geo_time_mean'] = test.set_index(
    ['geohash', 'timestamp']
).index.map(geo_time_mean)

# Fill missing values in test
global_mean = train['demand'].mean()

test['geo_time_mean'] = test['geo_time_mean'].fillna(global_mean)
test['geo_mean'] = test['geo_mean'].fillna(global_mean)
test['time_mean'] = test['time_mean'].fillna(global_mean)

In [ ]:
X = train.drop(columns=['demand', 'Index'])
y = train['demand']

test_X = test.drop(columns=['Index'])

cat_cols = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'timestamp'
]

for col in cat_cols:
    X[col] = X[col].astype('category')
    test_X[col] = test_X[col].astype('category')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

model = LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=12,
    num_leaves=64,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

r2 = r2_score(y_val, pred)

print("New R²:", r2)
print("Approx Score:", r2 * 100)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009089 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2052
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
New R²: 0.9930148347435778
Approx Score: 99.30148347435778


In [ ]:
model.fit(X, y)

test_pred = model.predict(test_X)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_pred
})

submission.to_csv('submission_v2.csv', index=False)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004647 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2053
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 13
[LightGBM] [Info] Start training from score 0.093942


In [ ]:
print(test_X.isnull().sum())

geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
geo_mean            0
time_mean           0
road_mean         324
geo_time_mean       0
dtype: int64


In [ ]:
# Fill categorical columns
test['RoadType'] = test['RoadType'].fillna(
    train['RoadType'].mode()[0]
)

test['Weather'] = test['Weather'].fillna(
    train['Weather'].mode()[0]
)

# Fill numerical column
test['Temperature'] = test['Temperature'].fillna(
    train['Temperature'].median()
)

In [ ]:
road_mean = train.groupby('RoadType')['demand'].mean()

test['road_mean'] = test['RoadType'].map(road_mean)

In [ ]:
test['geo_mean'] = test['geohash'].map(geo_mean)
test['time_mean'] = test['timestamp'].map(time_mean)
test['road_mean'] = test['RoadType'].map(road_mean)

test['geo_time_mean'] = test.set_index(
    ['geohash', 'timestamp']
).index.map(geo_time_mean)

test['geo_time_mean'] = test['geo_time_mean'].fillna(global_mean)

In [ ]:
test_X = test.drop(columns=['Index'])

for col in cat_cols:
    test_X[col] = test_X[col].astype('category')

In [ ]:
model.fit(X, y)
test_pred = model.predict(test_X)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005166 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2053
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 13
[LightGBM] [Info] Start training from score 0.093942


In [ ]:
X = train.drop(columns=['demand', 'Index'])
y = train['demand']

# categorical conversion
for col in cat_cols:
    X[col] = X[col].astype('category')

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("R²:", r2_score(y_val, pred))

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005854 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2052
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
R²: 0.9930148347435778


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_val, pred)
mae = mean_absolute_error(y_val, pred)
rmse = np.sqrt(mean_squared_error(y_val, pred))

print("R²   :", r2)
print("MAE  :", mae)
print("RMSE :", rmse)

NameError: name 'y_val' is not defined